# H3 — Transfer Learning with CNNs: Fashion-MNIST Live + CIFAR-10 Independent Activity

**References**
- Course CNN notebook: https://colab.research.google.com/github/serivan/DeepLearning/blob/master/PyTorch/12_deep_computer_vision_with_cnns.ipynb
- TorchVision pretrained models: https://docs.pytorch.org/vision/main/models.html

## Learning goals
By the end of this activity you should be able to:
1. distinguish **training from scratch**, **feature extraction**, and **fine-tuning**;
2. adapt a pretrained ImageNet CNN to a new classification task;
3. identify and replace the classification head of a pretrained architecture;
4. freeze/unfreeze parameters deliberately;
5. use different learning rates for pretrained and newly initialized parameters;
6. choose and justify a transfer-learning strategy for CIFAR-10;
7. compare frozen-feature extraction with partial fine-tuning and save the final model.

> Designing a CNN from scratch and hyperparameter tuning with Optuna are intentionally **not repeated here**: they belong to H2. H3 focuses on transfer learning.


In [ ]:
import copy
import time
import random
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

import torch
from torch import nn
from torch.utils.data import DataLoader, Subset
from torchvision import datasets, transforms, models

SEED = 42
random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)
if torch.cuda.is_available():
    torch.cuda.manual_seed_all(SEED)

device = torch.device(
    "cuda" if torch.cuda.is_available()
    else "mps" if torch.backends.mps.is_available()
    else "cpu"
)
pin_memory = device.type == "cuda"

print("PyTorch:", torch.__version__)
print("device:", device)


# Part A — Guided live: transfer learning on Fashion-MNIST

The previous CNN lab trained all convolutional filters **from scratch**. Here we keep Fashion-MNIST so that the dataset is familiar, but we change the learning strategy:

\[
\text{ImageNet pretrained CNN} \rightarrow \text{replace head} \rightarrow \text{freeze} \rightarrow \text{train head} \rightarrow \text{fine-tune selected layers}
\]

The purpose is not to obtain the best possible Fashion-MNIST score. Fashion-MNIST is intentionally very different from ImageNet; this makes **domain mismatch** visible and gives us something meaningful to discuss.


## A1 — Recall the CNN trained from scratch

A compact CNN previously studied on Fashion-MNIST can be written as one `nn.Sequential` pipeline. We will not retrain it here; use it only as a reference for the conceptual comparison.

**Discuss before running:**
- where are the features learned in this model?
- which parameters start from random initialization?
- if we move to a pretrained network, what changes?


In [ ]:
class SmallCNN(nn.Module):
    def __init__(self, n_classes=10):
        super().__init__()
        self.net = nn.Sequential(
            nn.Conv2d(1, 16, kernel_size=3, padding=1),
            nn.ReLU(),
            nn.MaxPool2d(2),
            nn.Conv2d(16, 32, kernel_size=3, padding=1),
            nn.ReLU(),
            nn.MaxPool2d(2),
            nn.Flatten(),
            nn.Linear(32 * 7 * 7, 64),
            nn.ReLU(),
            nn.Linear(64, n_classes),
        )

    def forward(self, x):
        return self.net(x)

reference_cnn = SmallCNN()
print(reference_cnn)
print("parameters:", f"{count_parameters(reference_cnn)[0]:,}" if 'count_parameters' in globals() else "run the helper cell below")


## A2 — Training and evaluation helpers

We deliberately use **two separate functions**:
- `train_one_epoch()` updates parameters;
- `evaluate()` never computes gradients or updates parameters.

This keeps the distinction explicit during the live session.


In [ ]:
def train_one_epoch(model, loader, criterion, optimizer):
    """Train for one epoch and return mean loss and accuracy."""
    model.train()
    running_loss = 0.0
    correct = 0
    total = 0

    for X, y in loader:
        X = X.to(device, non_blocking=True)
        y = y.to(device, non_blocking=True)

        optimizer.zero_grad(set_to_none=True)
        logits = model(X)
        loss = criterion(logits, y)
        loss.backward()
        optimizer.step()

        running_loss += loss.item() * X.size(0)
        correct += (logits.argmax(dim=1) == y).sum().item()
        total += y.numel()

    return running_loss / total, correct / total


@torch.no_grad()
def evaluate(model, loader, criterion):
    """Evaluate without updating model parameters."""
    model.eval()
    running_loss = 0.0
    correct = 0
    total = 0

    for X, y in loader:
        X = X.to(device, non_blocking=True)
        y = y.to(device, non_blocking=True)

        logits = model(X)
        loss = criterion(logits, y)

        running_loss += loss.item() * X.size(0)
        correct += (logits.argmax(dim=1) == y).sum().item()
        total += y.numel()

    return running_loss / total, correct / total


def count_parameters(model):
    total = sum(p.numel() for p in model.parameters())
    trainable = sum(p.numel() for p in model.parameters() if p.requires_grad)
    return total, trainable


## A3 — Pretrained ResNet-18: input requirements

We use **ResNet-18 pretrained on ImageNet**. The pretrained weights expect an ImageNet-style RGB input and their associated preprocessing.

Fashion-MNIST images are grayscale, so we must first convert them to RGB.

**Predict before coding:**
1. why can we not simply use the original 28×28 grayscale tensor?
2. what happens to computational cost when a 28×28 image is resized to an ImageNet-style input?
3. should the original ImageNet 1000-class head be kept?


In [ ]:
# TODO A1 — select the default pretrained ResNet-18 weights.
# weights = ...

# TODO A2 — build the transform:
# 1) convert Fashion-MNIST grayscale PIL images to RGB;
# 2) apply the preprocessing associated with the pretrained weights.
# transfer_transform = transforms.Compose([
#     ...,
#     ...,
# ])

# TODO A3 — create Fashion-MNIST train and test datasets using transfer_transform.
# fashion_train_full = ...
# fashion_test = ...

# Fixed train/validation split.
g = torch.Generator().manual_seed(SEED)
perm = torch.randperm(60_000, generator=g)
valid_idx = perm[:5_000].tolist()
train_idx = perm[5_000:].tolist()

# Keep the live experiment short.
LIVE_FAST = True
if LIVE_FAST:
    train_idx_live = train_idx[:4_000]
    valid_idx_live = valid_idx[:1_000]
    test_idx_live = list(range(1_000))
else:
    train_idx_live = train_idx
    valid_idx_live = valid_idx
    test_idx_live = list(range(10_000))

# TODO A4 — create Subset objects and DataLoaders.
# fashion_train = ...
# fashion_valid = ...
# fashion_test_live = ...
# fashion_train_loader = ...
# fashion_valid_loader = ...
# fashion_test_loader = ...

# TODO A5 — inspect one batch. Expected input shape for ResNet: [B, 3, H, W].


## A4 — Turn ResNet-18 into a Fashion-MNIST feature extractor

Feature-extraction workflow:
1. load pretrained weights;
2. freeze pretrained parameters;
3. replace the original classification head;
4. train only the new head.

**Important:** freezing parameters saves gradient/update work, but the forward pass through the backbone is still required.


In [ ]:
# TODO A6 — load pretrained ResNet-18.
# transfer_model = ...

# TODO A7 — freeze every pretrained parameter.
# for p in transfer_model.parameters():
#     ...

# TODO A8 — replace the 1000-class ImageNet head with a 10-class head.
# in_features = ...
# transfer_model.fc = ...
# transfer_model = transfer_model.to(device)

# TODO A9 — inspect total and trainable parameter counts.
# total_params, trainable_params = ...
# print(...)

# TODO A10 — run a forward pass on a few images and check logits.shape.


## A5 — Train only the new head

Use `CrossEntropyLoss`. The optimizer must initially receive **only the parameters of the new classification head**.

After the short run, compare training and validation accuracy. Do not expect ImageNet → Fashion-MNIST transfer to be perfect: the source and target domains are very different.


In [ ]:
# TODO A11 — define criterion and an optimizer for transfer_model.fc only.
# criterion_tl = ...
# optimizer_head = ...

# TODO A12 — train the head for 1–2 epochs using train_one_epoch() and evaluate().
# LIVE_EPOCHS = ...
# for epoch in range(LIVE_EPOCHS):
#     ...

# TODO A13 — evaluate the current model on fashion_test_loader.
# Keep this as a demonstration; in the CIFAR-10 challenge the test set will be reserved for the final model.


## A6 — From feature extraction to partial fine-tuning

Now allow the last residual stage (`layer4`) to adapt.

Use **discriminative learning rates**:
- a small learning rate for pretrained `layer4`;
- a larger learning rate for the newly initialized `fc` head.

**Discuss:** why is a smaller learning rate usually safer for pretrained parameters?


In [ ]:
# TODO A14 — unfreeze layer4.
# for p in transfer_model.layer4.parameters():
#     ...

# TODO A15 — build an optimizer with two parameter groups.
# optimizer_ft = torch.optim.AdamW([
#     {"params": ..., "lr": 1e-5},
#     {"params": ..., "lr": 1e-3},
# ], weight_decay=1e-4)

# TODO A16 — print the new trainable parameter count.

# OPTIONAL LIVE STEP — run one short fine-tuning epoch and compare validation accuracy.


### Live checkpoint
Be ready to explain these four statements in your own words:
1. **Training from scratch:** all useful visual filters must be learned from the target dataset.
2. **Feature extraction:** the pretrained representation is fixed and only the new head is trained.
3. **Fine-tuning:** some pretrained layers are allowed to adapt to the target domain.
4. **Domain similarity matters:** transfer is generally easier when source and target visual statistics/tasks are related.


# Part B — Independent CIFAR-10 transfer-learning activity

You will now apply the same ideas independently to CIFAR-10.

## Your task
Choose **one pretrained TorchVision CNN**. Recommended options are:
- ResNet-18;
- EfficientNet-B0;
- MobileNetV3-Small.

You may choose another ImageNet-pretrained CNN if you can correctly identify:
1. its weights object;
2. the preprocessing expected by those weights;
3. its classification head;
4. a suitable final feature block for partial fine-tuning.

You do **not** need to design another CNN from scratch in H3.


## B1 — Select a pretrained backbone and inspect it

Before writing training code, document your choice:

- model name;
- number of parameters;
- expected input preprocessing;
- name/location of the classification head;
- last feature block that you intend to fine-tune later.

For the recommended models:

| Model | Classification head | Suggested final feature block |
|---|---|---|
| ResNet-18 | `model.fc` | `model.layer4` |
| EfficientNet-B0 | `model.classifier[-1]` | `model.features[-1]` |
| MobileNetV3-Small | `model.classifier[-1]` | `model.features[-1]` |


In [ ]:
BACKBONE_NAME = "resnet18"   # TODO B1 — change if you choose another model

# TODO B2 — select the matching DEFAULT weights object and instantiate the model.
# Example for ResNet-18:
# weights = models.ResNet18_Weights.DEFAULT
# backbone = models.resnet18(weights=weights)

# TODO B3 — inspect weights.transforms() and print the model.
# eval_transform = weights.transforms()


## B2 — CIFAR-10 preprocessing for pretrained weights

Validation and test images must use the preprocessing expected by the selected pretrained weights.

For training, add a modest augmentation that preserves the class semantics. At minimum use a horizontal flip. You may add one other reasonable augmentation.

**Important:** the pretrained model expects ImageNet-style RGB inputs and normalization, so do not reuse the plain 32×32 transform from H2 without adapting it.


In [ ]:
# TODO B4 — build train_transform by combining a modest augmentation
# with preprocessing compatible with the selected weights.
# A simple pattern is:
# train_transform = transforms.Compose([
#     transforms.RandomHorizontalFlip(),
#     eval_transform,
# ])

# train_transform = ...

# TODO B5 — create two CIFAR-10 training datasets with identical examples:
# - one using train_transform;
# - one using eval_transform for validation.
# Also create the test dataset using eval_transform.

# TODO B6 — create a fixed 45k/5k train/validation split and DataLoaders.
# Use a smaller subset only if needed for a first debugging run.

# TODO B7 — inspect one batch. Confirm the shape expected by the backbone.


## B3 — Experiment 1: frozen feature extractor

First use the pretrained network as a **fixed feature extractor**:

1. freeze all pretrained parameters;
2. replace the ImageNet head with a 10-class head;
3. train only the new head;
4. evaluate validation accuracy;
5. record trainable parameters and training cost.

Do not unfreeze the backbone yet.


In [ ]:
def replace_head(model, model_name, n_classes=10):
    """Replace the classification head and return the new head module."""
    # TODO B8 — implement the branch corresponding to your selected model.
    # ResNet-18: model.fc
    # EfficientNet-B0 / MobileNetV3-Small: model.classifier[-1]
    raise NotImplementedError


# TODO B9 — freeze every existing parameter.

# TODO B10 — replace the classification head and move the model to device.
# new_head = replace_head(...)

# TODO B11 — verify that only the new head is trainable.
# print total/trainable parameter counts.

# TODO B12 — train only the new head for a small number of epochs.
# Use CrossEntropyLoss and an optimizer receiving new_head.parameters() only.

# Record the best validation accuracy and elapsed training time.


### B3 reflection

Explain briefly:
1. Why is the forward pass through the frozen backbone still computationally expensive?
2. Why can training only the head be useful when the target dataset is small?
3. Is ImageNet → CIFAR-10 a closer transfer problem than ImageNet → Fashion-MNIST? Why?


## B4 — Experiment 2: partial fine-tuning

Now allow the final pretrained feature block to adapt to CIFAR-10.

Use **discriminative learning rates**:
- smaller learning rate for pretrained parameters;
- larger learning rate for the new head.

Keep the experiment controlled: start from the same feature-extractor model (or reload its best checkpoint), unfreeze only the selected final block, and compare against Experiment 1.


In [ ]:
# TODO B13 — identify and unfreeze the selected final feature block.
# Suggested blocks:
#   ResNet-18: model.layer4
#   EfficientNet-B0: model.features[-1]
#   MobileNetV3-Small: model.features[-1]

# TODO B14 — create an optimizer with TWO parameter groups.
# Suggested starting point:
#   pretrained block LR: 1e-5 to 1e-4
#   new head LR:        1e-4 to 1e-3
# You may justify different values.

# TODO B15 — fine-tune for a few epochs and keep the best validation state.

# Record:
# - best validation accuracy;
# - trainable parameters;
# - elapsed training time.


## B5 — Independent extension: make one transfer-learning decision

Choose **one** of the following questions and test it with a controlled experiment:

**A. How much should be unfrozen?**  
Compare only the last block with a slightly deeper fine-tuning strategy.

**B. How important is the learning-rate ratio?**  
Change the ratio between pretrained-block LR and head LR.

**C. Does training augmentation still help after transfer?**  
Compare a minimal preprocessing pipeline with your augmented training pipeline.

**D. Does the classification head need regularization?**  
Add dropout to the new head where the selected architecture permits it.

State your hypothesis before running the experiment. Change only the factor needed to answer the selected question.


## B6 — Select the final transfer-learning strategy

Create a compact comparison table:

| Strategy | Trainable params | Best val. acc. | Training time | Notes |
|---|---:|---:|---:|---|
| Frozen backbone | ... | ... | ... | ... |
| Partial fine-tuning | ... | ... | ... | ... |
| Independent extension | ... | ... | ... | ... |

Select the final strategy **using validation results only** and justify the choice. A tiny validation improvement may not justify a large increase in computational cost.


## B7 — Final test evaluation and checkpoint

After the final transfer-learning strategy has been selected:

1. restore/retrain the selected configuration;
2. evaluate the CIFAR-10 test set **once**;
3. save the final checkpoint;
4. reload it into a fresh model and verify the result.

The checkpoint should contain at least:
- model `state_dict`;
- backbone name;
- selected strategy;
- relevant learning rates;
- test accuracy.


In [ ]:
FINAL_TRANSFER_PATH = "H3_cifar10_transfer_best.pt"

# TODO B16 — evaluate the selected final model on the test set once.

# TODO B17 — save checkpoint metadata + state_dict with torch.save(...).

# TODO B18 — reconstruct the selected pretrained architecture,
# replace the head, load state_dict, and verify the test result.


## B8 — Final discussion

Answer concisely:

1. Did partial fine-tuning improve over a frozen feature extractor? By how much?
2. Was the extra computational cost justified?
3. Which layers were adapted, and why were earlier layers kept frozen?
4. What evidence suggests that the ImageNet representation transferred successfully to CIFAR-10?
5. Compare conceptually with H2: when would you prefer a CNN trained from scratch, and when would you prefer transfer learning?

### Key distinction across H2 and H3
- **H2:** architecture design + regularization/data augmentation + Optuna, all trained from scratch.
- **H3:** reuse of pretrained visual representations + freezing + partial fine-tuning.
